In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

from study.data import cohort
from study.io import WORK, write_table
from study.statistics import loss_matrix, mean_draw_losses, risk_matrix

In [ ]:
# Compare hospital-specific information increments
predictions = pd.read_parquet(WORK / 'predictions/full')
predictions = predictions.loc[
    predictions.domain.eq('eicu')
    & predictions.source.eq('metavision')
    & predictions.support.eq('shared')
    & predictions.partition.eq('test')
]

metadata = cohort('eicu').query("partition == 'test'").sort_index()

# Estimate paired information increments within each test hospital
rows = []
for hospital, group in metadata.groupby('hospital_id'):
    ids = group.index

    for richer, simpler in [('R2', 'R1'), ('R3', 'R2')]:
        left = risk_matrix(predictions.loc[predictions.tier.eq(richer)], ids)
        right = risk_matrix(predictions.loc[predictions.tier.eq(simpler)], ids)
        delta = (
            mean_draw_losses(loss_matrix(group.in_hospital_mortality, left))
            - mean_draw_losses(loss_matrix(group.in_hospital_mortality, right))
        ).mean(axis = 1)

        rows.append(
            dict(
                hospital_id = hospital,
                contrast = richer + '_minus_' + simpler,
                patients = len(group),
                deaths = int(group.in_hospital_mortality.sum()),
                difference = delta.mean(),
                se = delta.std(ddof = 1) / np.sqrt(len(group)),
            )
        )

sites = pd.DataFrame(rows)

In [ ]:
# Summarise all hospitals and those meeting the 100/20/20 thresholds
rows = []
for patients, deaths, survivors in [(0, 0, 0), (100, 20, 20)]:
    eligible = sites.loc[
        sites.patients.ge(patients) & sites.deaths.ge(deaths) & (sites.patients - sites.deaths).ge(survivors)
    ]

    for contrast, group in eligible.groupby('contrast'):
        rows.append(
            dict(
                minimum_patients = patients,
                minimum_deaths = deaths,
                minimum_survivors = survivors,
                contrast = contrast,
                hospitals = len(group),
                patient_coverage = group.patients.sum() / len(metadata),
                proportion_benefiting = group.difference.lt(0).mean(),
                median = group.difference.median(),
                q10 = group.difference.quantile(0.1),
                q90 = group.difference.quantile(0.9),
            )
        )

write_table(WORK / 'results/hospital_effects.parquet', sites)
write_table(WORK / 'results/hospital_thresholds.parquet', pd.DataFrame(rows))

display(pd.DataFrame(rows).round(5))